# Mobility Matrix of a Rigid Assembly

Testing the `SoftBody` class with no degrees-of-freedom.

In [ ]:
import numpy as np
from softmobility import SoftBody

## Input with YAML format

In [2]:
yaml_data = """
design_names:
    - radius
    - xref
    
defaults:
    radius0: 1.
    radius1: 0.75
    radius2: 0.5
    xref0: 0.0
    xref1: 0.0
    xref2: 0.0

spheres:
  - # Sphere 0
    radius: radius0
    position: [xref0, xref1, xref2]
    
  - # Sphere 1
    radius: radius1
    position: [xref0 - 0.33 + radius0 + radius1, xref1, xref2]
    
  - # Sphere 2
    radius: radius2
    position: [xref0, xref1 + radius0 + radius2, xref2]    
"""

In [ ]:
# Creating SoftBody object
mybody = SoftBody(yaml_data)

  Found variables: radius0, radius1, radius2, xref0, xref1, xref2, 
    Sphere 0
      Radius: radius0
      Position: ['xref0', 'xref1', 'xref2']
      Orientation: ['0', '0', '0']
      Coupling matrix C_H:
[[], [], [], [], [], []]
      Coupling matrix C_K:
[[], [], [], [], [], []]
    Sphere 1
      Radius: radius1
      Position: ['radius0 + radius1 + xref0 - 0.33', 'xref1', 'xref2']
      Orientation: ['0', '0', '0']
      Coupling matrix C_H:
[[], [], [], [], [], []]
      Coupling matrix C_K:
[[], [], [], [], [], []]
    Sphere 2
      Radius: radius2
      Position: ['xref0', 'radius0 + radius2 + xref1', 'xref2']
      Orientation: ['0', '0', '0']
      Coupling matrix C_H:
[[], [], [], [], [], []]
      Coupling matrix C_K:
[[], [], [], [], [], []]


## Calculating the mobility matrix

In [ ]:
tensors = mybody.compute_tensors()

In [5]:
N = mybody.Nspheres
# Reshape M into a 3D array where each element is a 6x6 matrix representing one sphere's data
M0 = tensors.M @ mybody.compute_composition_of_forces()  # Mobility with forces/torques at the ref point x_0
M_blocks = M0.reshape(6, 6 * N).T.reshape(N, 6, 6)

# Check if all blocks are close to the first block with tolerance 1e-5
warning_printed = False
for i in range(1, N):
    if not np.isclose(M_blocks[i], M_blocks[0], atol=1e-5).all():
        warning_printed = True
        break

# Print a warning if any block differs significantly from the first one
if warning_printed:
    print("Warning: Matrix blocks of M are different (absolute tol 1e-5)")

# Compute the mean matrix by summing all blocks and dividing by Nspheres (implicitly through summation along axis=0)
M_mean = np.mean(M_blocks, axis=0)

In [6]:
print('Grand mobility matrix with forces/torques at the ref point x_0:')
print(M0)
print('\nReduced mobility matrix for the rigid assembly:')
print(M_mean)

Grand mobility matrix with forces/torques at the ref point x_0:
[[ 0.04614719 -0.00218714  0.          0.          0.          0.00394416
   0.04614719 -0.00218714  0.          0.          0.          0.00394416
   0.04614719 -0.00218714  0.          0.          0.          0.00394416]
 [-0.00218714  0.04645945  0.          0.          0.         -0.00682162
  -0.00218714  0.04645945  0.          0.          0.         -0.00682162
  -0.00218714  0.04645945  0.          0.          0.         -0.00682162]
 [ 0.          0.          0.04522518 -0.0058579   0.00828011  0.
   0.          0.          0.04522518 -0.0058579   0.00828011  0.
   0.          0.          0.04522518 -0.0058579   0.00828011  0.        ]
 [ 0.          0.         -0.0058579   0.02207929 -0.00139455  0.
   0.          0.         -0.0058579   0.02207929 -0.00139455  0.
   0.          0.         -0.0058579   0.02207929 -0.00139455  0.        ]
 [ 0.          0.          0.00828011 -0.00139455  0.01905566  0.
   0.     

## Assert the properties of the mobility matrix

In [7]:
# Check if the matrix is symmetric
assert np.allclose(M_mean, M_mean.T), "M_mean is not symmetric"

# Check if the matrix is positive semi-definite (all eigenvalues >= 0)
eigvals = np.linalg.eigvalsh(M_mean)
print("Eigenvalues of M:", eigvals)
assert np.all(eigvals >= 0), "M_mean is not positive definite"

print("The rigid mobility matrix is symmetric definite positive (as expected)")

Eigenvalues of M: [0.01375047 0.01663604 0.02075375 0.044184   0.04897034 0.05022256]
The rigid mobility matrix is symmetric definite positive (as expected)


## Hydrodynamic centers

Compute the coordinates of the hydrodynamic center of resistance and the hydrodynamic center of mobility, as defined in Kim and Karrila's book (Secs 5.2.2 and 5.3.2). Chosing these hydrodynamic centers as the origin of torques make the translation-rotation coupling symmetric for the resistance (resp. mobility) tensor. 

In [8]:
# Tensors to compute the mobility center
b = M_mean[3:, :3]
c = M_mean[3:, 3:]

# Tensors to compute the resistance center
resistance_matrix = np.linalg.inv(M_mean)
A = resistance_matrix[:3, :3]
B = resistance_matrix[3:, :3]

levicivita = np.array(
    [
        [[0.0, 0.0, 0.0], [0.0, 0.0, 1.0], [0.0, -1.0, 0.0]],
        [[0.0, 0.0, -1.0], [0.0, 0.0, 0.0], [1.0, 0.0, 0.0]],
        [[0.0, 1.0, 0.0], [-1.0, 0.0, 0.0], [0.0, 0.0, 0.0]],
    ]
)

# From Kim & Karila 5.2.2 and 5.3.2
traAmAinv = np.linalg.inv(np.trace(A) * np.eye(3) - A)
x_cr = 0.5 * np.einsum("ij,jkl,kl->i", traAmAinv, levicivita, B - B.transpose())
trcmcinv = np.linalg.inv(c - np.trace(c) * np.eye(3))
x_cm = 0.5 * np.einsum("ij,jkl,kl->i", trcmcinv, levicivita, b - b.transpose())

print("Hydrodynamic center of resistance:", x_cr)
print("Hydrodynamic center of mobility:", x_cm)

Hydrodynamic center of resistance: [-0.4274187  -0.24583946  0.        ]
Hydrodynamic center of mobility: [-0.42652943 -0.24468017  0.        ]


In [ ]:
# Assert the resistance tensor with x_cr the reference point
print("BEFORE change of coordinates:")
R_mean = np.linalg.inv(M_mean)
R_translation_rotation = R_mean[3:,:3]
print("Rotation-translation component of the resistance tensor:")
print(R_translation_rotation, "\n")


# Substract the calculated coordinates of the center of resistance to the coordinates of spheres
mybody.set_design_defaults(new_dict={"xref0": x_cr[0], "xref1": x_cr[1], "xref2": x_cr[2]})

# Compute the mean mobility matrix
tensors = mybody.compute_tensors()
M = tensors.M
M_mean = np.mean(M.reshape(6, 6 * mybody.Nspheres).T.reshape(mybody.Nspheres, 6, 6), axis=0)

# Compute the resistance matrix
R_mean = np.linalg.inv(M_mean)

print("\nAFTER change of coordinates:")
R_translation_rotation = R_mean[3:,:3]
print("Rotation-translation component of the resistance tensor:")
print(R_translation_rotation)

# Check if the matrix is symmetric
if np.allclose(R_translation_rotation, R_translation_rotation.T, atol=1E-5):
    print("R_mean is now symmetric")
else:
    raise ValueError("R_mean is not symmetric after change of coordinates")


BEFORE change of coordinates:
Rotation-translation component of the resistance tensor:
[[  0.         0.         5.919207]
 [  0.         0.       -10.32971 ]
 [ -5.516497  10.034225   0.      ]] 

OLD default param values: ['radius0', 'radius1', 'radius2', 'xref0', 'xref1', 'xref2'] [1.   0.75 0.5  0.   0.   0.  ]
NEW default param values: ['radius0', 'radius1', 'radius2', 'xref0', 'xref1', 'xref2'] [ 1.          0.75        0.5        -0.4274187  -0.24583946  0.        ]

AFTER change of coordinates:
Rotation-translation component of the resistance tensor:
[[ 0.          0.         -0.17011388]
 [ 0.          0.          0.25723836]
 [-0.17011444  0.25724176  0.        ]]
R_mean is now symmetric


In [ ]:
# Assert the mobility tensor with x_cm the reference point
print("BEFORE change of coordinates:")
M_translation_rotation = M_mean[3:,:3]
print("Rotation-translation component of the mobility tensor:")
print(M_translation_rotation, "\n")

# Substract the calculated coordinates of the center of resistance to the coordinates of spheres
mybody.set_design_defaults(new_dict={"xref0": x_cm[0], "xref1": x_cm[1], "xref2": x_cm[2]})

# Compute the mean mobility matrix
tensors = mybody.compute_tensors()
M = tensors.M
M_mean = np.mean(M.reshape(6, 6 * mybody.Nspheres).T.reshape(mybody.Nspheres, 6, 6), axis=0)

print("\nAFTER change of coordinates:")
M_translation_rotation = M_mean[3:,:3]
print("Rotation-translation component of the mobility tensor:")
print(M_translation_rotation)

# Check if the matrix is symmetric
if np.allclose(M_translation_rotation, M_translation_rotation.T, atol=1E-5):
    print("M_mean is now symmetric")
else:
    raise ValueError("M_mean is not symmetric after change of coordinates")


BEFORE change of coordinates:
Rotation-translation component of the mobility tensor:
[[ 0.          0.          0.00016612]
 [ 0.          0.         -0.00020748]
 [ 0.00012126 -0.00017509  0.        ]] 

OLD default param values: ['radius0', 'radius1', 'radius2', 'xref0', 'xref1', 'xref2'] [ 1.          0.75        0.5        -0.4274187  -0.24583946  0.        ]
NEW default param values: ['radius0', 'radius1', 'radius2', 'xref0', 'xref1', 'xref2'] [ 1.          0.75        0.5        -0.42652944 -0.24468017  0.        ]

AFTER change of coordinates:
Rotation-translation component of the mobility tensor:
[[ 0.          0.          0.00013928]
 [ 0.          0.         -0.00018891]
 [ 0.00013928 -0.00018891  0.        ]]
M_mean is now symmetric
